In [1]:
import pandas as pd
import os
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import accuracy_score
import numpy as np
import cv2 
from imagepreprocessing import process_img, resize_data

# Helper Methods

In [2]:
IMG_SHAPE = (100, 100)
SAVED_MODEL_FILE = "ripeness_model.sav"

In [4]:
# # Run once before training
# def resize_data():
#   data_path = 'data/'
#   for tvt in os.listdir(data_path): #iterate train val test split
#     ttest_path = os.path.join(data_path,tvt)
    
#     for rur in os.listdir(ttest_path): # iterate ripe unripe dirs
#       ripe_path = os.path.join(ttest_path, rur)
      
#       for name in os.listdir(ripe_path): # iterate image names
#         img_path = os.path.join(ripe_path, name)
        
#         img = cv2.imread(img_path)
#         if (img.shape != (IMG_SHAPE[0],IMG_SHAPE[1], 3)):
#           print(img.shape)
#           assert False
#         # img = cv2.resize(img, IMG_SHAPE)
#         # cv2.imwrite(img_path, img)


# resize_data()

In [5]:
# # Produce image features from image
# def process_img(img_path):
#   img = cv2.imread(img_path)
#   if (img.shape != (100, 100, 3)):
#     img = cv2.resize(img, (100, 100))
#   hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
  
#   counts = [[],[],[]]
#   # Count the occurances of each h, s, and v values
#   counts[0] = cv2.calcHist(hsv, [0], None, [179], [0, 179]).flatten('C')
#   counts[1] = cv2.calcHist(hsv, [1], None, [255], [0, 255]).flatten('C')
#   counts[2] = cv2.calcHist(hsv, [2], None, [255], [0, 255]).flatten('C')
  
#   # Flatten the overall array to be one dimension
#   return [item for channel in counts for item in channel]

# Load Train, Val, Test Data

In [6]:
data = {
  "train": [],
  "val": [],
  "test": [],
}
data_path = 'data/'
for tvt in os.listdir(data_path): #iterate train val test split
  ttest_path = os.path.join(data_path,tvt)
  for rur in os.listdir(ttest_path): # iterate ripe unripe dirs
    ripe_path = os.path.join(ttest_path, rur)
    for name in os.listdir(ripe_path): # iterate image names
      val = process_img(os.path.join(ripe_path, name))
      val.append(1 if rur == 'ripe' else 0)
      data[tvt].append(val)
      
for key in data.keys():
  data[key] = pd.DataFrame(data[key]).sample(frac=1) # create and shuffle df

In [7]:
X_train = data['train'].iloc[:,:-1]
y_train = data['train'].iloc[:,-1]
X_val = data['val'].iloc[:,:-1]
y_val = data['val'].iloc[:,-1]
X_test = data['test'].iloc[:,:-1]
y_test = data['test'].iloc[:,-1]
assert X_train.isna().sum().sum() == 0
assert X_val.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0
for target in y_val:
  if target not in [0,1]:
    print(target)

# Optimize Model

In [8]:
best_score =  0.9952628968253968
best_parm = {'gamma': 1, 'degree': 4, 'C': 10}

In [9]:
# param_grid = {
#   "degree": [3,4, 7, 8],
#   "gamma": 2**np.arange(0,3),
#   "C": 2**np.arange(0, 3),
#   }
# model = SVC(kernel='poly')
# clf = RandomizedSearchCV(model, param_grid, cv=10, n_jobs=-1, verbose=True)
# clf.fit(X_train, y_train)
# clf.best_score_
# if clf.best_score_ > best_score:
#   best_score = clf.best_score_
#   bst_parm = clf.best_params_
# print(f"Score: {best_score}, params: {best_parm}")
  

In [31]:

pd.concat((data['train'], data['val']), ignore_index=True)
len(X_train)

636

In [34]:
final_model = SVC(kernel='poly', gamma=1, degree=4, C=10)
final_model.fit(pd.concat((X_train, X_val)), pd.concat((y_train, y_val)))
accuracy_score(y_test, final_model.predict(X_test))

1.0

# Testing

In [12]:
test_samples = []
base_test_dir = 'fruits-360-original-size/Test/'
for apple_type in os.listdir(base_test_dir):
  inner_dir = os.path.join(base_test_dir, apple_type)
  for name in os.listdir(inner_dir):
    test_samples.append(os.path.join(inner_dir, name))

test_samples = pd.DataFrame(test_samples)

In [52]:
counts = []
pos = []
for _ in range(10):
  counter = 0
  pred = 0
  while pred == 0:
    img_path = test_samples.sample(1).values[0][0]
    pred = final_model.predict(np.array(process_img(img_path)).reshape(1,-1))
    counter += 1
  pos.append(img_path)
  counts.append(counter)
print(sum(counts)/len(counts))
print(pos)

289.6
['fruits-360-original-size/Test/apple_red_yellow_1\\r0_203.jpg', 'fruits-360-original-size/Test/apple_braeburn_1\\r0_211.jpg', 'fruits-360-original-size/Test/apple_red_yellow_1\\r0_211.jpg', 'fruits-360-original-size/Test/apple_braeburn_1\\r0_199.jpg', 'fruits-360-original-size/Test/apple_braeburn_1\\r0_203.jpg', 'fruits-360-original-size/Test/carrot_1\\r0_87.jpg', 'fruits-360-original-size/Test/apple_braeburn_1\\r0_211.jpg', 'fruits-360-original-size/Test/apple_red_yellow_1\\r0_203.jpg', 'fruits-360-original-size/Test/apple_braeburn_1\\r0_203.jpg', 'fruits-360-original-size/Test/carrot_1\\r0_91.jpg']


# Dump Model

In [38]:
import pickle
with open(SAVED_MODEL_FILE, "wb") as save_file:
  pickle.dump(final_model, save_file)

## Test Dump

In [53]:
with open(SAVED_MODEL_FILE, "rb") as load_file:
  loaded_model = pickle.load(load_file)
loaded_model.predict(np.array(process_img('fruits-360-original-size/Test/apple_red_yellow_1\\r0_203.jpg')).reshape(1,-1))

array([1], dtype=int64)

# Test Prediction Class

In [3]:
from predict_ripe import RipenessPredictor
estim = RipenessPredictor(SAVED_MODEL_FILE)
print(estim.predict_ripe(cv2.imread('fruits-360-original-size/Test/apple_red_yellow_1\\r0_203.jpg')))
print(estim.predict_ripe(cv2.imread('fruits-360-original-size/Test/apple_golden_1/r0_103.jpg')))

True
False
